# Marktindex: hoe bewegen occasionprijzen over de tijd?

De prijs van een auto in de toekomst hangt af van twee dingen:

1. **Afschrijving**: de auto wordt ouder en rijdt meer kilometers. Dat leert het model in `prijsvoorspelling.ipynb`.
2. **De markt**: worden alle occasions duurder of goedkoper? Dat meten we hier.

In dit notebook:

1. **Vergelijkbaarheid** van de twee bronnen: Wayback (pagina 1 van de zoekresultaten, maart 2023 – dec 2025)
   en de scrape van oktober 2026 (het volledige aanbod).
2. **Marktindex per maand**: de prijs van een "standaardauto" met vaste leeftijd, kilometerstand en model.
   Dat is de time series die we straks doortrekken naar de toekomst.
3. **Controle** met de CBS-prijsindex voor tweedehands auto's.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

SCRAPE_DATUM = "2026-10-05"   # einde van de cars.csv-scrape (zelfde als prijsvoorspelling.ipynb)
MAX_LEEFTIJD = 20

## 1. Data inladen

Zelfde filters als in `prijsvoorspelling.ipynb`: alleen NL, en van de Wayback-data alleen `page_sort == "standard"`.
De nieuwprijs is hier niet nodig, want we kijken naar de verkoopprijs zelf.

In [ ]:
scrape = pd.read_csv("../webscrape/data/cars.csv")
scrape["snapshot_date"] = SCRAPE_DATUM
scrape["bron"] = "scrape_2026"

wb = pd.read_csv("../webscrape/data/wayback_cars.csv")
wb = wb[(wb["country"] == "NL") & (wb["page_sort"] == "standard")].copy()
wb["bron"] = "wayback"

kolommen = ["bron", "snapshot_date", "brand", "model", "year", "mileage_km", "fuel_type", "transmission", "price_eur"]
d = pd.concat([scrape[kolommen], wb[kolommen]], ignore_index=True)
d = d.dropna(subset=["model", "year", "mileage_km"])

d["snapshot_date"] = pd.to_datetime(d["snapshot_date"])
d["maand"] = d["snapshot_date"].dt.to_period("M").astype(str)
d["observation_year"] = d["snapshot_date"].dt.year
d["age"] = d["observation_year"] - d["year"].astype(int)
d["brand_model"] = d["brand"] + " " + d["model"]

d = d[(d["age"] >= 0) & (d["age"] <= MAX_LEEFTIJD)]
d = d[(d["mileage_km"] < 1_000_000) & (d["price_eur"] < 500_000)]

print(d["bron"].value_counts())
print("Wayback-maanden:", d.loc[d["bron"] == "wayback", "maand"].nunique())

## 2. Zijn de bronnen vergelijkbaar?

Wayback is pagina 1 van de standaardsortering van AutoScout24, inclusief gepromote advertenties.
De 2026-scrape is het volledige aanbod. Als de Wayback-auto's systematisch anders zijn (nieuwer, minder km,
duurder), dan zit dat verschil straks in de marktindex en lijkt het alsof de markt veranderd is.

Eerst: hoe zien de auto's er gemiddeld uit per bron? Voor Wayback alleen de laatste 6 maanden (jul – dec 2025),
die liggen het dichtst bij oktober 2026.

In [ ]:
recent = d[(d["bron"] == "scrape_2026") | (d["snapshot_date"] >= "2025-07-01")]

recent.groupby("bron").agg(
    auto_s=("price_eur", "size"),
    mediaan_prijs=("price_eur", "median"),
    mediaan_leeftijd=("age", "median"),
    mediaan_km=("mileage_km", "median"),
    aandeel_automaat=("transmission", lambda s: (s == "automatic").mean()),
    aandeel_elektrisch=("fuel_type", lambda s: (s == "electric").mean()),
).round(2)

Dan een eerlijkere vergelijking: dezelfde auto's naast elkaar. Per combinatie van merk + model, leeftijd en brandstof
die in **beide** bronnen voorkomt: de verhouding van de mediaanprijzen (2026 / Wayback eind 2025).
Een verhouding rond 1,0 betekent dat de bronnen vergelijkbaar zijn; het verschil is dan ~10 maanden marktontwikkeling.

In [ ]:
cel = ["brand_model", "age", "fuel_type"]
per_bron = recent.groupby(cel + ["bron"])["price_eur"].agg(["median", "size"]).unstack("bron").dropna()
per_bron = per_bron[per_bron[("size", "wayback")] >= 2]

per_bron["verhouding"] = per_bron[("median", "scrape_2026")] / per_bron[("median", "wayback")]
gewicht = per_bron[("size", "wayback")]

print(f"{len(per_bron)} vergelijkbare combinaties, {int(gewicht.sum())} Wayback-auto's")
print(f"Mediane verhouding 2026 / Wayback:  {per_bron['verhouding'].median():.3f}")
print(f"Gewogen gemiddelde (log):           {np.exp(np.average(np.log(per_bron['verhouding']), weights=gewicht)):.3f}")

plt.figure(figsize=(8, 4))
plt.hist(np.log(per_bron["verhouding"]), bins=40)
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("log(prijs 2026 / prijs Wayback), per combinatie")
plt.ylabel("Aantal combinaties")
plt.title("Zelfde model, leeftijd en brandstof: 2026 vs Wayback eind 2025")
plt.show()

## 3. Marktindex per maand

Regressie op de log-prijs:

`log(prijs) = maand + merk_model + leeftijd + brandstof + transmissie + log(km)`

Het effect van `maand` is de marktindex: hoeveel duurder of goedkoper dezelfde auto in die maand was.
Leeftijd is een categorie (0, 1, 2, ... jaar), zodat de afschrijvingscurve elke vorm mag hebben.
Basis = oktober 2026 = 100.

Kanttekening: tussen december 2025 en oktober 2026 wisselt ook de bron. Het laatste punt bevat dus naast de markt
ook het bronverschil uit stap 2. Daarom tonen we ook een index op **alleen Wayback-data** (basis = maart 2023 = 100):
die komt uit één bron en is daardoor de zuiverste trend.

In [ ]:
def marktindex(data, basis_maand):
    """Index per maand (basis_maand = 100) uit een regressie op log(prijs), plus het aantal auto's per maand."""
    maand_dummies = pd.get_dummies(data["maand"]).drop(columns=basis_maand).astype(float)

    categorisch = data[["brand_model", "fuel_type", "transmission"]].assign(age=data["age"].astype(str))
    encoder = OneHotEncoder(drop="first", handle_unknown="ignore")
    X = sparse.hstack([
        sparse.csr_matrix(maand_dummies.values),
        encoder.fit_transform(categorisch),
        sparse.csr_matrix(np.log1p(data[["mileage_km"]].values)),
    ]).tocsr()

    reg = LinearRegression().fit(X, np.log(data["price_eur"]))
    effect = pd.Series(reg.coef_[:maand_dummies.shape[1]], index=maand_dummies.columns)
    effect[basis_maand] = 0.0

    index = pd.DataFrame({"index": 100 * np.exp(effect)}).sort_index()
    index["n"] = data["maand"].value_counts()
    return index


index_alles = marktindex(d, basis_maand=d["maand"].max())
index_wayback = marktindex(d[d["bron"] == "wayback"], basis_maand=d.loc[d["bron"] == "wayback", "maand"].min())

index_alles.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = pd.PeriodIndex(index_alles.index, freq="M").to_timestamp()
axes[0].plot(x, index_alles["index"], marker="o")
axes[0].axhline(100, color="grey", linewidth=0.8)
axes[0].set_title("Marktindex, beide bronnen (okt 2026 = 100)")
axes[0].set_ylabel("Index (nominaal)")

x = pd.PeriodIndex(index_wayback.index, freq="M").to_timestamp()
axes[1].plot(x, index_wayback["index"], marker="o", color="tab:orange")
axes[1].axhline(100, color="grey", linewidth=0.8)
axes[1].set_title("Marktindex, alleen Wayback (mrt 2023 = 100)")

for ax in axes:
    ax.grid(True)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Controle met CBS

CBS meet de prijsontwikkeling van tweedehands auto's in de CPI (`cpi_used_cars` in `cpi.csv`, per jaar).
Als onze index ongeveer dezelfde kant op beweegt, meten we echt de markt en niet alleen ruis of bronverschillen.
Beide reeksen worden omgezet naar 2023 = 100.

In [ ]:
cpi = pd.read_csv("cpi.csv", index_col="year")

per_jaar = index_alles.assign(jaar=pd.PeriodIndex(index_alles.index, freq="M").year)
# Gewogen naar aantal auto's per maand; 2026 is één maand (de scrape)
per_jaar["gewogen"] = per_jaar["index"] * per_jaar["n"]
eigen = per_jaar.groupby("jaar")["gewogen"].sum() / per_jaar.groupby("jaar")["n"].sum()
eigen = 100 * eigen / eigen.loc[2023]

cbs = 100 * cpi["cpi_used_cars"] / cpi.loc[2023, "cpi_used_cars"]

vergelijking = pd.DataFrame({"eigen_index": eigen, "cbs_tweedehands": cbs.loc[eigen.index]}).round(1)
vergelijking["verschil"] = vergelijking["eigen_index"] - vergelijking["cbs_tweedehands"]
vergelijking

## 5. Opslaan

`marktindex.csv` wordt straks gebruikt om de markt door te trekken naar de toekomst (scenario's)
en om het model in `prijsvoorspelling.ipynb` op een marktneutrale prijs te trainen.

In [ ]:
uit = index_alles.rename(columns={"index": "index_beide_bronnen"})
uit["index_alleen_wayback"] = index_wayback["index"]
uit.index.name = "maand"
uit.round(2).to_csv("marktindex.csv")
uit.round(1).tail()

## Hoe lees je dit?

- **Stap 2, verhouding rond 1,0**: de bronnen zijn vergelijkbaar en `index_beide_bronnen` is bruikbaar.
  Wijkt hij flink af (bijv. meer dan ±10%), gebruik dan de trend uit `index_alleen_wayback` en behandel het
  laatste punt (okt 2026) met voorzichtigheid.
- **Stap 4**: beweegt de eigen index dezelfde kant op als CBS, dan meet hij de markt. Een groot verschil wijst op
  een bronverschil of te weinig data in sommige maanden (kijk naar kolom `n`).
- Deze index geeft ~3,5 jaar marktgeschiedenis, in een periode waarin de prijzen na de coronapiek daalden.
  Trek hem daarom niet blind lineair door, maar werk met scenario's (dalend / gelijk / stijgend).